# BT — Cooperative PHY Jamming with MARL
**Run this notebook on a GPU cluster (student cluster / Colab / Euler Jupyter).**

Cells are ordered — run top to bottom once.

In [ ]:
# ── Cell 1: Environment check ─────────────────────────────────────────
import subprocess, sys

# GPU
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                         '--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', result.stdout.strip() if result.returncode == 0 else 'NOT FOUND — will use CPU')
print('Python:', sys.version.split()[0])

# Check key packages
for pkg in ['torch', 'sionna', 'gymnasium', 'stable_baselines3', 'matplotlib']:
    try:
        mod = __import__(pkg)
        print(f'  {pkg}: {mod.__version__}')
    except ImportError:
        print(f'  {pkg}: NOT INSTALLED')

In [ ]:
# ── Cell 2: Install missing packages (skip if already installed) ───────
!pip install -q sionna>=2.0 stable-baselines3 gymnasium matplotlib

In [ ]:
# ── Cell 3: Clone repo and set up path ────────────────────────────────
import os

REPO_URL = 'https://github.com/rahul2605-ux/BT.git'
REPO_DIR = '/tmp/BT'          # change if you prefer a different location

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

# Add repo root to Python path (no packaging needed)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

os.chdir(REPO_DIR)   # run scripts from repo root
print('Working directory:', os.getcwd())

# Create output dirs
for d in ['models/stage1', 'models/ctde', 'logs/stage1', 'logs/ctde',
          'plots/stage1', 'plots/ctde_n3', 'plots/multi_agent']:
    os.makedirs(d, exist_ok=True)

In [ ]:
# ── Cell 4: Sanity check — one env step with Sionna ───────────────────
from core.config import EnvironmentConfig
from envs.wireless_env import WirelessEnv

env = WirelessEnv(EnvironmentConfig(channel_mode='sionna', seed=42))
obs, _ = env.reset()
_, reward, _, _, info = env.step(env.action_space.sample())
env.close()

print(f'Sionna env OK | obs={obs.shape} | SINR={info["mean_sinr_db"]:.1f} dB')

In [ ]:
# ── Cell 5: Barrage baseline + multi-agent sweep ──────────────────────
# Fast fixed-strategy simulation — no training needed, ~1 min
%run simulations/simple_barrage.py
%run simulations/multi_agent_sim.py --channel sionna --out plots/multi_agent

In [ ]:
# ── Cell 6a: Train Stage 1 — single jammer PPO ────────────────────────
# ~30–60 min on a GPU depending on cluster speed
# Generates plots/stage1/ automatically when done

TOTAL_STEPS = 500_000   # increase to 1M for cleaner curves if time allows
N_ENVS      = 4         # parallel envs — increase if you have RAM

%run simulations/train_stage1.py \
    --total-steps {TOTAL_STEPS} \
    --n-envs {N_ENVS} \
    --channel-mode sionna

In [ ]:
# ── Cell 6b: Train Stage 1c — CTDE team (3 jammers) ───────────────────
# ~60–90 min on GPU. Can run in parallel with 6a if cluster allows.
# Generates plots/ctde_n3/ automatically when done

N_JAMMERS   = 3
TOTAL_STEPS = 1_000_000

%run simulations/train_ctde.py \
    --n-jammers {N_JAMMERS} \
    --total-steps {TOTAL_STEPS} \
    --n-envs {N_ENVS} \
    --channel-mode sionna

In [ ]:
# ── Cell 7: Display all results inline ────────────────────────────────
from IPython.display import Image, display
import glob

plot_dirs = {
    'Barrage baseline + multi-agent': 'plots/multi_agent',
    'Stage 1 — single jammer PPO':    'plots/stage1',
    'Stage 1c — CTDE team (N=3)':     'plots/ctde_n3',
}

for title, d in plot_dirs.items():
    pngs = sorted(glob.glob(f'{d}/*.png'))
    if not pngs:
        print(f'No plots yet in {d}')
        continue
    print(f'\n─── {title} ───')
    for p in pngs:
        print(os.path.basename(p))
        display(Image(p, width=800))

In [ ]:
# ── Cell 8: Print summary numbers ─────────────────────────────────────
import numpy as np

for name, path in [('Stage1',   'logs/stage1/metrics.npz'),
                   ('CTDE N=3', 'logs/ctde/metrics.npz')]:
    if not os.path.exists(path):
        print(f'{name}: not trained yet')
        continue
    d = np.load(path)
    sinr = d['ep_sinr']
    # Compare first 10% vs last 10% of training
    n = len(sinr)
    early = sinr[:n//10].mean()
    late  = sinr[-n//10:].mean()
    print(f'{name}: SINR  early={early:.1f} dB  →  late={late:.1f} dB  '
          f'(improvement: {early-late:.1f} dB)')